In [ ]:
"Hello from google colab"

'Hello from google colab'

In [3]:
# ============================================================
# QWEN3-TTS — COLAB SETUP
# ============================================================

!pip install -U qwen-tts soundfile gradio


  Using cached gradio-6.32.0-py3-none-any.whl.metadata (16 kB)
INFO: pip is looking at multiple versions of gradio to determine which version is compatible with other requirements. This could take a while.
  Using cached gradio-6.31.0-py3-none-any.whl.metadata (16 kB)
  Using cached gradio-6.30.0-py3-none-any.whl.metadata (16 kB)
  Using cached gradio-6.29.1-py3-none-any.whl.metadata (16 kB)
  Using cached gradio-6.29.0-py3-none-any.whl.metadata (16 kB)
  Using cached gradio_client-2.7.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached gradio-6.28.0-py3-none-any.whl.metadata (16 kB)
  Using cached gradio-6.27.0-py3-none-any.whl.metadata (17 kB)
  Using cached gradio_client-2.7.0-py3-none-any.whl.metadata (7.1 kB)
  Using cached gradio-6.26.0-py3-none-any.whl.metadata (17 kB)
  Using cached gradio_client-2.6.1-py3-none-any.whl.metadata (7.1 kB)
INFO: pip is still looking at multiple versions of gradio to determine which version is compatible with other requirements. This could take a w

In [4]:
# ============================================================
# GPU CHECK
# ============================================================

import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )
else:
    raise RuntimeError(
        "GPU not detected. Go to Runtime → Change runtime type → GPU."
    )

PyTorch: 2.14.1+cpu
CUDA available: False


RuntimeError: GPU not detected. Go to Runtime → Change runtime type → GPU.

In [5]:
# ============================================================
# LOAD QWEN3-TTS
# ============================================================

import torch
import soundfile as sf

from qwen_tts import Qwen3TTSModel

MODEL_ID = "Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice"

print("Loading Qwen3-TTS...")
print("Model:", MODEL_ID)

model = Qwen3TTSModel.from_pretrained(
    MODEL_ID,
    device_map="cuda:0",
    dtype=torch.bfloat16,
)

print("\n✅ Qwen3-TTS loaded successfully!")

# Show available speakers/languages from the actual model
print("\nSupported speakers:")
print(model.get_supported_speakers())

print("\nSupported languages:")
print(model.get_supported_languages())


    If you do not have SoX, proceed here:
     - - - http://sox.sourceforge.net/ - - -

    If you do (or think that you should) have SoX, double-check your
    path variables.
    



********
********
 
Loading Qwen3-TTS...
Model: Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]


✅ Qwen3-TTS loaded successfully!

Supported speakers:
['aiden', 'dylan', 'eric', 'ono_anna', 'ryan', 'serena', 'sohee', 'uncle_fu', 'vivian']

Supported languages:
['auto', 'chinese', 'english', 'french', 'german', 'italian', 'japanese', 'korean', 'portuguese', 'russian', 'spanish']


In [6]:
# ============================================================
# QUICK TEST
# ============================================================

test_text = """
This is a quick test of Qwen three TTS.
If you're hearing this clearly, everything is working correctly.
"""

wavs, sr = model.generate_custom_voice(
    text=test_text.strip(),
    language="English",
    speaker="Ryan",
    instruct="Natural, confident, engaging YouTube Shorts narrator."
)

test_wav = "/content/qwen_test.wav"

sf.write(test_wav, wavs[0], sr)

print("✅ Generated:", test_wav)
print("Sample rate:", sr)

from IPython.display import Audio, display

display(Audio(test_wav))

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


✅ Generated: /content/qwen_test.wav
Sample rate: 24000


In [1]:
# ============================================================
# QWEN3-TTS SHORTS STUDIO — COLAB VERSION
# ============================================================

import os
import subprocess
import time
import gradio as gr


OUTPUT_DIR = "/content/qwen_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SPEAKERS = [
    "Ryan",
    "Aiden",
    "Vivian",
    "Serena",
    "Sohee",
    "Ono_Anna",
    "Uncle_Fu",
    "Dylan",
    "Eric",
]

DEFAULT_INSTRUCTION = (
    "Natural, engaging, confident YouTube Shorts narrator. "
    "Clear pronunciation, expressive delivery, medium-fast pace."
)


def generate_tts(text, speaker, instruction):

    if not text or not text.strip():
        raise gr.Error("Please enter some text.")

    text = text.strip()

    if not instruction or not instruction.strip():
        instruction = DEFAULT_INSTRUCTION

    timestamp = time.strftime("%Y%m%d_%H%M%S")

    wav_path = os.path.join(
        OUTPUT_DIR,
        f"qwen_{speaker}_{timestamp}.wav"
    )

    mp3_path = os.path.join(
        OUTPUT_DIR,
        f"qwen_{speaker}_{timestamp}.mp3"
    )

    print(f"Generating with {speaker}...")

    start = time.time()

    # Qwen3-TTS generation
    wavs, sr = model.generate_custom_voice(
        text=text,
        language="English",
        speaker=speaker,
        instruct=instruction.strip(),
    )

    # Save WAV
    sf.write(wav_path, wavs[0], sr)

    # Convert WAV → MP3
    subprocess.run(
        [
            "ffmpeg",
            "-y",
            "-i",
            wav_path,
            "-codec:a",
            "libmp3lame",
            "-q:a",
            "2",
            mp3_path,
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=True,
    )

    elapsed = time.time() - start

    status = (
        f"✅ Generated successfully\n"
        f"Voice: {speaker}\n"
        f"Characters: {len(text):,}\n"
        f"Generation time: {elapsed:.1f}s"
    )

    print(status)

    return wav_path, wav_path, mp3_path, status


# ============================================================
# UI
# ============================================================

with gr.Blocks(title="Qwen3-TTS Shorts Studio") as demo:

    gr.Markdown(
        """
        # 🎙️ Qwen3-TTS Shorts Studio

        Generate voiceovers for YouTube Shorts.
        """
    )

    text_input = gr.Textbox(
        label="Script",
        placeholder="Paste your Shorts script here...",
        lines=10,
    )

    speaker_input = gr.Dropdown(
        label="Voice",
        choices=SPEAKERS,
        value="Ryan",
    )

    instruction_input = gr.Textbox(
        label="Voice / Style",
        value=DEFAULT_INSTRUCTION,
        lines=4,
    )

    generate_button = gr.Button(
        "🎙️ Generate Voice",
        variant="primary",
    )

    audio_output = gr.Audio(
        label="Generated Audio",
        type="filepath",
    )

    status_output = gr.Textbox(
        label="Status",
        interactive=False,
    )

    with gr.Row():
        wav_download = gr.File(label="WAV")
        mp3_download = gr.File(label="MP3")

    generate_button.click(
        fn=generate_tts,
        inputs=[
            text_input,
            speaker_input,
            instruction_input,
        ],
        outputs=[
            audio_output,
            wav_download,
            mp3_download,
            status_output,
        ],
    )


# ============================================================
# LAUNCH
# ============================================================

print("Starting Qwen3-TTS UI...")
print("Please wait for the Gradio link...\n")

demo.launch(
    share=True,
    debug=False,
)

Starting Qwen3-TTS UI...
Please wait for the Gradio link...

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5514d2f99c386c39a2.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
